# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

d

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
from pyspark.sql import functions as F

bookings_df = spark.read.csv("/Volumes/test_flight_data/default/pgexercises/bookings.csv", header=True, inferSchema=True)
members_df = spark.read.csv("/Volumes/test_flight_data/default/pgexercises/members.csv", header=True, inferSchema=True)
facilities_df = spark.read.csv("/Volumes/test_flight_data/default/pgexercises/facilities.csv", header=True, inferSchema=True)

transformed_df = (
    bookings_df
    .filter(
        (F.col("starttime") >= "2012-09-01") &
        (F.col("starttime") < "2012-10-01")
    )
    .groupBy("facid")
    .agg(F.sum("slots").alias("Total Slots"))
    .orderBy("Total Slots")
)

transformed_df.show()

output_path = "/Volumes/test_flight_data/default/pgexercises/fachoursbymonth_parquet"

(
    transformed_df
    .write
    .mode("overwrite")
    .parquet(output_path)
)

print(f"Parquet files written to: {output_path}")


+-----+-----------+
|facid|Total Slots|
+-----+-----------+
|    5|        122|
|    3|        422|
|    7|        426|
|    8|        471|
|    6|        540|
|    2|        570|
|    1|        588|
|    0|        591|
|    4|        648|
+-----+-----------+

Parquet files written to: /Volumes/test_flight_data/default/pgexercises/fachoursbymonth_parquet


## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
from pyspark.sql import functions as F

bookings_df = spark.read.csv("/Volumes/test_flight_data/default/pgexercises/bookings.csv", header=True, inferSchema=True)
members_df = spark.read.csv("/Volumes/test_flight_data/default/pgexercises/members.csv", header=True, inferSchema=True)
facilities_df = spark.read.csv("/Volumes/test_flight_data/default/pgexercises/facilities.csv", header=True, inferSchema=True)


transformed_df = (
    bookings_df.alias("b")
    .join(members_df.alias("m"), F.col("b.memid") == F.col("m.memid"), "inner")
    .join(facilities_df.alias("f"), F.col("b.facid") == F.col("f.facid"), "inner")
    .filter(F.col("f.name").like("Tennis Court%"))
    .select(
        F.concat(F.col("m.firstname"), F.lit(" "), F.col("m.surname")).alias("member"),
        F.col("f.name").alias("facility")
    )
    .distinct()
    .orderBy("member", "facility")
)

display(transformed_df)

(
    transformed_df
    .write
    .mode("overwrite")
    .format("delta")
    .partitionBy("facility")
    .saveAsTable("threejoin_delta")
)

member,facility
Anne Baker,Tennis Court 1
Anne Baker,Tennis Court 2
Burton Tracy,Tennis Court 1
Burton Tracy,Tennis Court 2
Charles Owen,Tennis Court 1
Charles Owen,Tennis Court 2
Darren Smith,Tennis Court 2
David Farrell,Tennis Court 1
David Farrell,Tennis Court 2
David Jones,Tennis Court 1


## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F

# -------------------------
# EXTRACT
# -------------------------

url = "https://alpha-vantage.p.rapidapi.com/query"

file_path = "/Volumes/test_flight_data/default/pgexercises/secret.txt"

with open(file_path, "r") as f:
    secret_value = f.read().strip()

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": secret_value
}

symbols = ["GOOG", "AAPL", "MSFT", "TSLA"]

all_rows = []

for symbol in symbols:
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "datatype": "json",
        "outputsize": "compact"
    }

    response = requests.get(url, headers=headers, params=querystring)
    response.raise_for_status()

    data = response.json()

    time_series = data.get("Time Series (Daily)", {})

    for trade_date, values in time_series.items():
        all_rows.append({
            "symbol": symbol,
            "trade_date": trade_date,
            "open": float(values["1. open"]),
            "high": float(values["2. high"]),
            "low": float(values["3. low"]),
            "close": float(values["4. close"]),
            "volume": int(values["5. volume"])
        })

stock_prices_df = pd.DataFrame(all_rows)

stock_prices_df["trade_date"] = pd.to_datetime(stock_prices_df["trade_date"])
stock_prices_df = stock_prices_df.sort_values(by=["symbol", "trade_date"], ascending=[True, False])

spark_df = spark.createDataFrame(stock_prices_df)

# -------------------------
# TRANSFORM
# Find weekly max closing price for each company
# -------------------------

spark_df = spark_df.withColumn("trade_date", F.to_date("trade_date"))

weekly_max_df = (
    spark_df
    .withColumn("week", F.date_trunc("week", F.col("trade_date")))
    .groupBy("symbol", "week")
    .agg(F.max("close").alias("max_closing_price"))
    .orderBy("symbol", "week")
)

display(weekly_max_df)

# -------------------------
# LOAD
# Partition by symbol
# Save into managed Delta table
# -------------------------

(
    weekly_max_df
    .write
    .mode("overwrite")
    .format("delta")
    .partitionBy("symbol")
    .saveAsTable("max_closing_price_weekly")
)

print("Loaded into managed table: max_closing_price_weekly")

symbol,week,max_closing_price
AAPL,2025-11-17T00:00:00.000Z,271.49
AAPL,2025-11-24T00:00:00.000Z,278.85
AAPL,2025-12-01T00:00:00.000Z,286.19
AAPL,2025-12-08T00:00:00.000Z,278.78
AAPL,2025-12-15T00:00:00.000Z,274.61
AAPL,2025-12-22T00:00:00.000Z,273.81
AAPL,2025-12-29T00:00:00.000Z,273.76
AAPL,2026-01-05T00:00:00.000Z,267.26
AAPL,2026-01-12T00:00:00.000Z,261.05
AAPL,2026-01-19T00:00:00.000Z,248.35


Loaded into managed table: max_closing_price_weekly


## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
jdbc_hostname = "hh-pgsql-public.ebi.ac.uk"
jdbc_port = 5432
jdbc_database = "pfmegrnargs"
jdbc_url = f"jdbc:postgresql://{jdbc_hostname}:{jdbc_port}/{jdbc_database}"

connection_properties = {
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "driver": "org.postgresql.Driver"
}

query = "(SELECT * FROM rna LIMIT 100) AS rna_100"

rna_df = spark.read.jdbc(
    url=jdbc_url,
    table=query,
    properties=connection_properties
)

display(rna_df)

rna_df.write.mode("overwrite").saveAsTable("rna_100_records")


id,upi,timestamp,userstamp,crc64,len,seq_short,seq_long,md5
16293310,URS0000F89DBE,2019-12-02T13:19:27.359Z,rnacen,DF55DAF98BAE3649,253,TACGGAGGATGCAAGCGTTATCCGGAATGATTGGGCGTAAAGCGTCCGCAGGTGGCTGTGTAAGTCTGCTGTTAAAGAGTGAGGCTCAACCTCATAAAAGCAGTGGAAACTACACAGCTAGAGTGCGTTCGGGGCAGAGGGAATTCCTGGTGTAGCGGTGAAATGCGTAGAGATCAGGAAGAACACCGGTGGCGAAAGCGTTCTGCTAGACCTGTACTGACACTGAGGGACGAAAGCTAGGGGAGCGAATGGG,null,0d4f27476cb980c23cec7a11a6151643
16293311,URS0000F89DBF,2019-12-02T13:19:27.359Z,rnacen,3E4DC14D5F5274A3,253,TACGAAGGGGGCTAGCGTTGCTCGGAATCACTGGGCGTAAAGGGTGCGTAGGCGGGTCTTTAAGTCAGGGGTGAAATCCTGGAGCTCAACTCCAGAACTGCCTTTGATACTGAAGATCTTGAGTTCGGGAGAGGTGAGTGGAACTGCGAGTGTAGAGGTGAAATTCGTAGATATTCGCAAGAACACCAGTGGCGAAGGCGGCTCACTGGCCCGATACTGACGCTGAGGCGCGAAGGCGTGGGGAGCGAACGGG,null,0d4f2cc3679298b822dcad8a50d794e4
16293312,URS0000F89DC0,2019-12-02T13:19:27.359Z,rnacen,F0C41137905993D1,396,ATACGTAGGTGGCAAGCGTTGTCCGGAATTATTGGGCGTAAAGCGCATGTAGGCGGTGCCTTAAGTCTGTCGTGAAACTGCGGGGCCTAACCCCGTATGGCGATGGAAACTGTGGCCCTTGAGTGCAGGAGAGGAAAGGGGAACTCCCAGTGTAGCGGTTAAATGCGTAGATATTGGGAAGAACACCGGTGGCGAAGGCGCGTTTCTGGACTGTGACTGACGCTGAGATGCGAAAGCCAGGGTAGCGAACGGGATTAGATACCCCGGTAGTCCTGGCCGTAAACGATGGGTACTAGGTGTGGGAGGTATCGACCCCTTCCGTGCCGGAGTTAACGCAATAAGTACCCCACCTGGGGAGTACGGCCGCAAGGCTTAAACTTAAAGGAATTGACGGGG,null,0d4f2e356e09639f05996107935ebb26
16293313,URS0000F89DC1,2019-12-02T13:19:27.359Z,rnacen,12E9361C36A39D12,469,AGGGTTTGATTCTGGCTCAGAACGAACGCTGGCGGCATGCCTAACACATGCAAGTCGAACGAAGGCTTCGGCCTTAGTGGCGCACGAGTGCGTAACGCGTGGGAATCTGCACTTGGGTTCGGAATAACAGCGGGAAACTGCTGCTAATACCGGATGAAGACGAAAGTCCAAAGATTTATCGCCTGAGGATGAGCCCGCGTTGGATTAGGTAGTTGGTGGGGTAAAGGCCTACCAAGCCGACGATCCATAGCTGGTCTGAGAGGATGATCAGCCACACTGGGACTGAGACACGGCCCAGACTCCTACGGGAGGCAGCAGTGGGGAATATTGGACAATGGGCGAAAGCCTGATCCAGCAATGCCGCGTGAGTGATGAAGTCCTTAGGTTTGTAAAGCTCTTTTACCCGGGATGATAATGACAGTACCGGGAGAATAAGCCCCGGCTAACTCCGGGCCAGCAGCCGCGGTAA,null,0d4f33b676c03b9dfd371ecf5196728e
16293314,URS0000F89DC2,2019-12-02T13:19:27.359Z,rnacen,6C593B7B442A5DE8,253,TACAGAGACTGCAAGCGTTATTCGGATTCACTGGGCGTAAAGGGTGCGCAGGCGGCCAAGTGTGTGAGGCGTGAAAGCCCGGGGCTTAACCCCGGAATTGCACCTCAAACTACTTGGCTAGAGCATTGGAGAGGGTAGCAGAATTCACGGTGTGGCAGTGAAATGCGTAGATATCGTGAGGAATACCAGAGGCGAAGGCGGCTACCTGGACAATTGCTGACGCTCAGGCACGAAAGCGTGGGGAGCAAAAGGG,null,0d4f34900f97c9fb19d874d2151d7889
16293315,URS0000F89DC3,2019-12-02T13:19:27.359Z,rnacen,36D4C7FB05CC73DC,469,AGGGTTTGATTATGGCTCAGGAAGATAACAGTAGCTGAGCTGGATGGAACCAGATCACGTATCTGGGCTGAGTCCTGGTGGTGCCTGCACCAACGCCTCGGGGAGCTTGTGGCTGCTCGCTGTGGGAAGTGTGTGGTTCTTCCATGCCCAGCCTTCCCAGCGGGGACTGAAGACTGGGAACCAGTACATATAGTACAGGTATTTTTAGCCTAAAGATTTTCATTTCATTATAGAATTGGCTAGTCTTTGCTGTCTCCTCACTTTTGAACACTAGTGGGTCCTAGACTGCATGGCACCTTGATTTAAGTCATATACGTATCAAAAGCTAGTAACCCTGAGATCATTCAAGCACAGAAGCCTTTGGTCTTGCAGGGTGGCTTCACTGCTAAAAGGAGAGGACAGGCTTTCACCAAACTGCCTTTGCTGGGATCACATTGCTGGCCAAGCCCTGTGCCAGCAGCCGCGGTAA,null,0d4f3505ebd87e7a49bb77aa47c1e361
16293316,URS0000F89DC4,2019-12-02T13:19:27.359Z,rnacen,4FF2DEAD4523342C,253,GACGAACCGTGCGAACGTTGTTCGGAATCACTGGGCTTAAAGGGCGCGCAGGCGGCCTGCCAAGTCCGGGGTGAAATCCTCCAGCTTAACTGGAGAAGTGCCTTGGATACTGGCGAGCTCGAGCGAGGCAGGGGTGATGGGAACTTCCGGTGGAGCGGTGAAATGCGTTGATATCGGAAGGAACGCCGGTGGCGAAAGCGCATCACTGGTCCTCTTCTGACGCTGAGGCGCGAAAGCCAGGGGAGCAAACGGG,null,0d4f37c6c5cceefd29842af9851610ba
16293317,URS0000F89DC5,2019-12-02T13:19:27.359Z,rnacen,A0E5F006CBE846FF,229,TGCGTAGGCGGAGGATTAAGTCAGTGGTGAAATCTCACAGCTCAACTGTGAAACTGCCATTGAAACTGATTTTCTTGAATACGGTTGAGGTAGGCGGAATATGTTATGTAGCGGTGAAATGCTTAGATATAACATAGAACACCAATTGCGAAGGCAGCTTACTAAGCCGTTATTGACGCTGAGGCACGAAAGCGTGGGGAGCGAACAGGATTAGATACCCTGGTAGTCC,null,0d4f3abfa32a40704ccde874c468ef9a
16293318,URS0000F89DC6,2019-12-02T13:19:27.359Z,rnacen,BAA74B0884DFFE80,550,AAAACTAATGGGGAATCTTGCGCAATGGACGAAAGTCTGACGCAGCGACGCCGCGTGGGGGATGAAGGCTTTCGGGTTGTAAACCCCTGTTGCCCGGGACGAACTTCTGCTTTCGAGCAGATTGACGGTACCGGGTGAGGAAGCACCGGCTAACTCTGTGCCAGCAGCCGCGGTAATACAGAGGGTGCGAGCGTTGTCCGGAATCACTGGGCGTAAAGGGCGCGTAGGTGGCGCGACAAGTCAGTGGTGAAAGTTCGACGCTCAACGTCGAGTCGGCCACTGATACTGTTGGGGTTGAGCACTGTGGAGGTTAATGGAATTCCGGGGGTAGCGGTGGAATGCGTAGAGATCCGGAAGAACACCGGTGGC